# 🔢 تطبيق تصنيف الأرقام المكتوبة يدويًا
# Advanced Handwritten Digit Recognition Application

## مقدمة | Introduction

هذا الـ Notebook يوضح كيفية بناء نموذج متقدم لتصنيف الأرقام المكتوبة يدويًا باستخدام:
- **Binarisation**: تحويل الصور إلى صور ثنائية (أسود وأبيض)
- **K-means Clustering**: تقسيم البيانات إلى مجموعات
- **معالجة مسبقة محسّنة**: تطبيع وتحسين الصور

This notebook demonstrates how to build an advanced model for handwritten digit classification using:
- **Binarisation**: Converting images to binary (black and white)
- **K-means Clustering**: Dividing data into clusters
- **Enhanced Preprocessing**: Normalization and image enhancement

## 1️⃣ استيراد المكتبات | Import Libraries

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

print('✓ تم استيراد جميع المكتبات بنجاح')
print('✓ All libraries imported successfully')

## 2️⃣ تحميل البيانات | Load Data

In [ ]:
# تحميل بيانات MNIST
digits = load_digits()
X = digits.data
y = digits.target
images = digits.images

print(f'✓ عدد العينات: {X.shape[0]}')
print(f'✓ Number of samples: {X.shape[0]}')
print(f'✓ حجم الصورة: {images[0].shape}')
print(f'✓ Image size: {images[0].shape}')
print(f'✓ عدد الميزات: {X.shape[1]}')
print(f'✓ Number of features: {X.shape[1]}')

## 3️⃣ عرض عينات من البيانات | Display Data Samples

In [ ]:
# عرض أول 16 صورة
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('عينات من بيانات MNIST | MNIST Data Samples', fontsize=14, fontweight='bold')

for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f'الرقم: {y[i]} | Digit: {y[i]}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 4️⃣ تطبيق Binarisation | Apply Binarisation

In [ ]:
def binarize_image(image, threshold=None):
    """
    تطبيق binarisation على الصورة
    Apply binarisation to image
    """
    # تطبيع الصورة
    if image.max() > 1:
        image = image / 255.0
    
    # حساب العتبة تلقائيًا
    if threshold is None:
        threshold = image.mean()
    
    # تطبيق binarisation
    binary_image = (image > threshold).astype(np.uint8)
    
    return binary_image

# تطبيق binarisation على جميع الصور
X_binary = np.zeros_like(X)

for i in range(len(X)):
    img = X[i].reshape(8, 8)
    binary_img = binarize_image(img)
    X_binary[i] = binary_img.flatten()

print(f'✓ تم تطبيق binarisation على {len(X)} صورة')
print(f'✓ Applied binarisation to {len(X)} images')

## 5️⃣ مقارنة الصور قبل وبعد Binarisation | Compare Before and After Binarisation

In [ ]:
# عرض مقارنة بين الصور الأصلية والثنائية
fig, axes = plt.subplots(4, 4, figsize=(12, 10))
fig.suptitle('مقارنة: الصور الأصلية vs الصور الثنائية | Comparison: Original vs Binary Images', 
             fontsize=14, fontweight='bold')

for i in range(8):
    # الصور الأصلية
    ax = axes[i // 4, i % 4 * 2]
    ax.imshow(images[i], cmap='gray')
    ax.set_title(f'الأصلية | Original')
    ax.axis('off')
    
    # الصور الثنائية
    ax = axes[i // 4, i % 4 * 2 + 1]
    binary_img = X_binary[i].reshape(8, 8)
    ax.imshow(binary_img, cmap='gray')
    ax.set_title(f'ثنائية | Binary')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 6️⃣ تطبيع البيانات | Normalize Data

In [ ]:
# تطبيع البيانات الثنائية
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_binary)

print(f'✓ شكل البيانات المعالجة: {X_scaled.shape}')
print(f'✓ Processed data shape: {X_scaled.shape}')
print(f'✓ المتوسط: {X_scaled.mean():.4f}')
print(f'✓ Mean: {X_scaled.mean():.4f}')
print(f'✓ الانحراف المعياري: {X_scaled.std():.4f}')
print(f'✓ Standard deviation: {X_scaled.std():.4f}')

## 7️⃣ تدريب نموذج K-means | Train K-means Model

In [ ]:
# تدريب K-means
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10, max_iter=300)
kmeans.fit(X_scaled)
predictions = kmeans.labels_

print(f'✓ تم تدريب K-means بنجاح')
print(f'✓ K-means training completed')
print(f'✓ Inertia: {kmeans.inertia_:.2f}')
print(f'✓ عدد التكرارات: {kmeans.n_iter_}')
print(f'✓ Number of iterations: {kmeans.n_iter_}')

## 8️⃣ ربط المجموعات بالأرقام | Map Clusters to Digits

In [ ]:
# ربط المجموعات بالأرقام الفعلية
cluster_to_digit = {}

for cluster_id in range(10):
    mask = predictions == cluster_id
    if mask.sum() > 0:
        most_common_digit = np.bincount(y[mask]).argmax()
        cluster_to_digit[cluster_id] = most_common_digit

print('✓ جدول ربط المجموعات بالأرقام:')
print('✓ Cluster-to-Digit mapping table:')
print('-' * 40)
print(f'{'معرف المجموعة':<20} {'الرقم المقابل':<20}')
print(f'{'Cluster ID':<20} {'Mapped Digit':<20}')
print('-' * 40)

for cluster_id in sorted(cluster_to_digit.keys()):
    digit = cluster_to_digit[cluster_id]
    print(f'{cluster_id:<20} {digit:<20}')

print('-' * 40)

## 9️⃣ تقييم النموذج | Evaluate Model

In [ ]:
# حساب دقة النموذج
predicted_digits = np.array([cluster_to_digit.get(pred, -1) for pred in predictions])
accuracy = (predicted_digits == y).sum() / len(y)

print(f'✓ دقة النموذج: {accuracy*100:.2f}%')
print(f'✓ Model Accuracy: {accuracy*100:.2f}%')

# عرض الدقة لكل رقم
print('\n✓ الدقة لكل رقم | Accuracy per digit:')
print('-' * 40)
print(f'{'الرقم':<10} {'الدقة':<30}')
print(f'{'Digit':<10} {'Accuracy':<30}')
print('-' * 40)

for digit in range(10):
    mask = y == digit
    if mask.sum() > 0:
        digit_accuracy = (predicted_digits[mask] == y[mask]).sum() / mask.sum()
        print(f'{digit:<10} {digit_accuracy*100:>6.2f}%')

print('-' * 40)

## 🔟 عرض النتائج | Display Results

In [ ]:
# عرض النتائج
fig, axes = plt.subplots(4, 4, figsize=(12, 10))
fig.suptitle('نتائج التصنيف | Classification Results', fontsize=14, fontweight='bold')

for i in range(16):
    ax = axes[i // 4, i % 4]
    ax.imshow(images[i], cmap='gray')
    
    cluster_id = predictions[i]
    predicted_digit = cluster_to_digit.get(cluster_id, -1)
    true_digit = y[i]
    
    color = 'green' if predicted_digit == true_digit else 'red'
    title = f'C:{cluster_id} | P:{predicted_digit} | T:{true_digit}'
    ax.set_title(title, color=color, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.show()

print('✓ الأخضر = تنبؤ صحيح | Green = Correct prediction')
print('✓ الأحمر = تنبؤ خاطئ | Red = Incorrect prediction')

## 1️⃣1️⃣ اختبار التنبؤ على صور جديدة | Test Predictions on New Images

In [ ]:
def predict_digit(image, apply_binarization=True):
    """
    التنبؤ برقم من صورة
    Predict digit from image
    """
    # تحويل الصورة إلى 8x8
    if image.shape != (8, 8):
        image = cv2.resize(image, (8, 8))
    
    # تطبيق binarisation
    if apply_binarization:
        image = binarize_image(image)
    
    # تسطيح وتطبيع
    image_flat = image.flatten().reshape(1, -1)
    image_scaled = scaler.transform(image_flat)
    
    # التنبؤ
    cluster_id = kmeans.predict(image_scaled)[0]
    predicted_digit = cluster_to_digit.get(cluster_id, -1)
    
    # حساب الثقة
    distance = np.linalg.norm(image_scaled[0] - kmeans.cluster_centers_[cluster_id])
    confidence = 1 / (1 + distance)
    
    return predicted_digit, cluster_id, confidence

# اختبار على عينات
print('✓ اختبار التنبؤ على عينات مختلفة:')
print('✓ Testing predictions on different samples:')
print('-' * 70)
print(f'{'الفهرس':<10} {'الرقم الحقيقي':<15} {'الرقم المتنبأ':<15} {'الثقة':<15} {'صحيح؟':<10}')
print(f'{'Index':<10} {'True Digit':<15} {'Predicted':<15} {'Confidence':<15} {'Correct?':<10}')
print('-' * 70)

test_indices = [0, 100, 500, 1000, 1500]
for idx in test_indices:
    true_digit = y[idx]
    image = images[idx]
    
    predicted_digit, cluster_id, confidence = predict_digit(image, apply_binarization=True)
    
    is_correct = '✓' if predicted_digit == true_digit else '✗'
    print(f'{idx:<10} {true_digit:<15} {predicted_digit:<15} {confidence*100:>6.1f}%{'':<8} {is_correct:<10}')

print('-' * 70)

## 1️⃣2️⃣ الخلاصة والنتائج | Summary and Results

In [ ]:
print('\n' + '='*70)
print('📊 ملخص النتائج | Results Summary')
print('='*70)

summary = {
    'إجمالي العينات': len(X),
    'عدد المجموعات': 10,
    'دقة النموذج': f'{accuracy*100:.2f}%',
    'Inertia': f'{kmeans.inertia_:.2f}',
    'عدد التكرارات': kmeans.n_iter_,
    'حجم الصورة': '8×8 بكسل'
}

for key, value in summary.items():
    print(f'{key:<20} : {value}')

print('='*70)
print('✓ تم إكمال التحليل بنجاح!')
print('✓ Analysis completed successfully!')
print('='*70)